# Lesson 4.4: Selecting, Filtering, and Aggregating

**Analyst:** John Ko
**Date:** 11/05/26
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Practice column selection, boolean filtering, and
groupby aggregation on the 200,000-row FinFlow dataset. Compute the
overall churn rate. Preview the per-tier churn breakdown.

Friday checkpoint with Hannah: churn rate by plan tier (Practice Exercise
of this lesson) and churn rate by signup cohort (Lesson 4.6).

In [1]:
# Imports
import pandas as pd
import numpy as np

#Set options for Pandas to display all columns and all flot will be rounded by second decimal 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-customers-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

print(f"Loaded {len(df):,} rows from finflow-customers-200k.csv")
print(f"Shape: {df.shape}")

Loaded 200,000 rows from finflow-customers-200k.csv
Shape: (200000, 16)


## Selecting columns

Three syntactic forms: single-column bracket (returns Series),
multi-column bracket (returns DataFrame), dot notation (limited).


In [2]:
# Single column, returns Series
fees = df['monthly_fee']
print("Type of df['monthly_fee']:", type(fees).__name__)
print("Shape:", fees.shape)
print()

# Multiple columns, returns DataFrame
focused = df[['customer_id', 'plan_tier', 'monthly_fee', 'active']]
print("Type of multi-column selection:", type(focused).__name__)
print("Shape:", focused.shape)
print()

#Inspect the focused DataFrame
focused.head()

Type of df['monthly_fee']: Series
Shape: (200000,)

Type of multi-column selection: DataFrame
Shape: (200000, 4)



,customer_id,plan_tier,monthly_fee,active
0,FF-00001,Basic,9.99,True
1,FF-00002,Pro,29.99,True
2,FF-00003,Basic,9.99,True
3,FF-00004,Basic,9.99,True
4,FF-00005,Basic,9.99,True


## Filtering rows by condition

Boolean Series filters DataFrames. Create the condition, pass it into
brackets, get the matching rows back.


In [3]:
# All Pro-tier customers
pro_customers = df[df['plan_tier'] == 'Pro']
print(f"Pro customers: {len(pro_customers):,}")

# All active customers
active_customers = df[df['active']]
print(f"Active customers: {len(active_customers):,}")

# All churned customers
churned_customers = df[~df['active']]
print(f"Churned customers: {len(churned_customers):,}")

# Total 
print(f"Total: {len(pro_customers) + 0:,} Pro, "
      f"{len(active_customers):,} active, "
      f"{len(churned_customers):,} churned out of {len(df)}")

Pro customers: 60,229
Active customers: 160,049
Churned customers: 39,951
Total: 60,229 Pro, 160,049 active, 39,951 churned out of 200000


In [4]:
# Pro customers who are active
pro_active = df[(df['plan_tier'] == 'Pro') & (df['active'])]
print(f"Pro AND active: {len(pro_active):,}")

# Pro OR Enterprise customers (two ways)
pro_or_enterprise_v1 = df[(df['plan_tier'] == 'Pro') | (df['plan_tier'] == 'Enterprise')]
pro_or_enterprise_v2 = df[df['plan_tier'].isin(['Pro', 'Enterprise'])]
print(f"Pro OR Enterprise (via |): {len(pro_or_enterprise_v1):,}")
print(f"Pro OR Enterprise (via isin): {len(pro_or_enterprise_v2):,}")
# Both numbers should be identical.

# Active customers signed up in the last 6 months
from datetime import datetime, timedelta, date
six_months_ago = pd.Timestamp("2026-01-01") - pd.Timedelta(days=180)
AS_OF_DATE = date(2026, 1, 1)          # the same as-of date the dataset was built with
recent_active = df[(df['active']) & (df['signup_date'] >= six_months_ago)]
print(f"Active AND signed up in last 180 days: {len(recent_active):,}")
six_months_ago = pd.Timestamp(AS_OF_DATE) - pd.Timedelta(days=180)

Pro AND active: 48,129
Pro OR Enterprise (via |): 80,119
Pro OR Enterprise (via isin): 80,119
Active AND signed up in last 180 days: 39,831


In [5]:
# DELIBERATELY BROKEN: missing parentheses
try:
    broken = df[df['plan_tier'] == 'Pro' & df['active']]
    print(f"Broken filter result: {len(broken):,}")
except Exception as e:
    print(f"Error class: {type(e).__name__}")
    print(f"Error message: {e}")

Error class: TypeError
Error message: Cannot perform 'rand_' with a dtyped [bool] array and scalar of type [bool]


**Observation:** Forgetting the parentheses produces a TypeError that does
not name the actual problem. The fix is always to wrap each condition in its
own pair of parens before combining with `&`, `|`, or `~`.


## loc and iloc

loc is label-based. iloc is position-based. On a freshly-loaded DataFrame
they look identical; after sorting or filtering they diverge.



In [6]:
# Looking at the same row with both accessors on the freshly-loaded df
print("df.iloc[0]:")
print(df.iloc[0][['customer_id', 'plan_tier', 'total_sessions']])
print()
print("df.loc[0] (the customer orginally at index 0):")
print(df.loc[0][['customer_id', 'plan_tier', 'total_sessions']])

df.iloc[0]:
customer_id       FF-00001
plan_tier            Basic
total_sessions         134
Name: 0, dtype: object

df.loc[0] (the customer orginally at index 0):
customer_id       FF-00001
plan_tier            Basic
total_sessions         134
Name: 0, dtype: object


In [7]:
# Sort by total_sessions descending
df_sorted = df.sort_values('total_sessions', ascending=False)

print("df_sorted.iloc[0] (the highest-session customer):")
print(df_sorted.iloc[0][['customer_id', 'plan_tier', 'total_sessions']])
print()
print("df_sorted.loc[0] (the customer originally at index 0):")
print(df_sorted.loc[0][['customer_id', 'plan_tier', 'total_sessions']])


df_sorted.iloc[0] (the highest-session customer):
customer_id       FF-96188
plan_tier              Pro
total_sessions         400
Name: 96187, dtype: object

df_sorted.loc[0] (the customer originally at index 0):
customer_id       FF-00001
plan_tier            Basic
total_sessions         134
Name: 0, dtype: object


**Observation:** On the sorted DataFrame, iloc and loc return different
customers. iloc gives the highest-session customer (position 0 in the new
order). loc gives FF-00001 (the original label 0, wherever it landed in the
sort). This is the bug class senior analysts watch for. Rule of thumb: use
iloc when you mean "the first row by current order." Use loc when you mean
"the row with this specific label or condition."

In [8]:
# loc can take boolean filters AND specific columns at the same time
active_with_subset = df.loc[df['active'], ['customer_id', 'plan_tier', 'monthly_fee']]
print(f"Shape: {active_with_subset.shape}")
active_with_subset.head()

Shape: (160049, 3)


,customer_id,plan_tier,monthly_fee
0,FF-00001,Basic,9.99
1,FF-00002,Pro,29.99
2,FF-00003,Basic,9.99
3,FF-00004,Basic,9.99
4,FF-00005,Basic,9.99


## Overall aggregations and the churn rate


In [9]:
# Basic aggregations on the monthly_fee column
print("Total monthly fee (all customers):", df['monthly_fee'].sum())
print("Mean monthly fee:", df['monthly_fee'].mean())
print("Median monthly fee:", df['monthly_fee'].median())
print()

# Plan tier counts
print("Plan tier distribution:")
print(df['plan_tier'].value_counts())
print()

# Overall churn rate via boolean mean trick
churn_rate = (~df['active']).mean() * 100
print(f"Overall churn rate: {churn_rate:.2f}%")

Total monthly fee (all customers): 6961988.9
Mean monthly fee: 34.8099445
Median monthly fee: 9.99

Plan tier distribution:
plan_tier
Basic         119881
Pro            60229
Enterprise     19890
Name: count, dtype: int64

Overall churn rate: 19.98%


## groupby: the split-apply-combine model

Group by plan_tier. Compute the per-tier active rate. Convert to churn rate.
This is the move that answers Hannah's first Friday question.


In [10]:
# Per-tier mean of the active column (proportion active)
active_rate_by_tier = df.groupby('plan_tier')['active'].mean()
print("Active rate by plan tier:")
print(active_rate_by_tier)
print()

# Convert to churn rate by tier
churn_rate_by_tier = (1 - df.groupby('plan_tier')['active'].mean()) * 100
print("Churn rate by plan tier:")
print(churn_rate_by_tier)

Active rate by plan tier:
plan_tier
Basic        0.80
Enterprise   0.80
Pro          0.80
Name: active, dtype: float64

Churn rate by plan tier:
plan_tier
Basic        19.93
Enterprise   19.88
Pro          20.09
Name: active, dtype: float64


**Observation:** Churn rate varies meaningfully across plan tiers (or it
doesn't, depending on which generation script produced the data). Either
way, the groupby pattern is the same: split by tier, apply the aggregation
to each group, combine into a single result indexed by tier.


In [11]:
# Multiple metrics per tier in one DataFrame
summary_by_tier = df.groupby('plan_tier').agg({
    'monthly_fee': 'sum',
    'total_sessions': 'mean',
    'avg_session_duration_minutes': 'mean',
    'support_tickets_opened': 'mean'
})
print("Per-tier summary:")
print(summary_by_tier)

Per-tier summary:
            monthly_fee  total_sessions  avg_session_duration_minutes  \
plan_tier                                                               
Basic        1197611.19          172.81                          7.90   
Enterprise   3958110.00          173.46                          7.90   
Pro          1806267.71          173.26                          7.88   

            support_tickets_opened  
plan_tier                           
Basic                         2.41  
Enterprise                    2.40  
Pro                           2.39  


In [12]:
# Group by two columns: plan_tier and active
two_way = df.groupby(['plan_tier', 'active'])['monthly_fee'].mean()
print("Mean monthly fee by tier and active state:")
print(two_way)

Mean monthly fee by tier and active state:
plan_tier   active
Basic       False      9.99
            True       9.99
Enterprise  False    199.00
            True     199.00
Pro         False     29.99
            True      29.99
Name: monthly_fee, dtype: float64


In [13]:
# Unstack to make active a column dimension
two_way_unstacked = df.groupby(['plan_tier', 'active'])['monthly_fee'].mean().unstack()
print("Same data, unstacked:")
print(two_way_unstacked)

Same data, unstacked:
active      False  True 
plan_tier               
Basic        9.99   9.99
Enterprise 199.00 199.00
Pro         29.99  29.99


In [14]:
# Custom function: churn rate as a percentage
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

# Apply via agg() alongside other metrics
final_summary = df.groupby('plan_tier').agg({
    'customer_id': 'count',
    'monthly_fee': 'sum',
    'active': churn_rate_pct,
    'total_sessions': 'mean'
})
final_summary.columns = ['customer_count', 'total_mmr', 'churn_rate_pct', 'avg_sessions']
print("Final per-tier summary:")
print(final_summary)

Final per-tier summary:
            customer_count  total_mmr  churn_rate_pct  avg_sessions
plan_tier                                                          
Basic               119881 1197611.19           19.93        172.81
Enterprise           19890 3958110.00           19.88        173.46
Pro                  60229 1806267.71           20.09        173.26


##  A short AI demonstration (Tier 1)

In [15]:
median_by_tier = df.groupby('plan_tier')['monthly_fee'].median()
print(median_by_tier)

plan_tier
Basic          9.99
Enterprise   199.00
Pro           29.99
Name: monthly_fee, dtype: float64


**AI Use Note:** Used Claude to confirm the median-by-group idiom. Output
matched the groupby pattern I'd already seen. Verified by running.
Tier 1 disclosure.